In [0]:
import os

path = "/Volumes/databrickssidibe/datasidibe/nouveau/"

files = dbutils.fs.ls(path)

for file in files:
    file_path = file.path
    file_name = file.name.split(".")[0]  # nom sans extension
    
    df = spark.read.option("header", True).option("inferSchema", True).csv(file_path)
    
    df.write.mode("overwrite").saveAsTable(f"bronze.{file_name}")
    
    print(f"Table bronze.{file_name} créée")

In [0]:
%sql
show tables in bronze

In [0]:
%sql
drop table bronze.ventes;





In [0]:
df_clients = spark.read.csv("/Volumes/databrickssidibe/datasidibe/nouveau/clients.csv",header=True,inferSchema=True)
display(df_clients)

df_commerciaux = spark.read.csv("/Volumes/databrickssidibe/datasidibe/nouveau/commerciaux.csv",header=True,inferSchema=True)
display(df_commerciaux)

df_concessionnaires = spark.read.csv("/Volumes/databrickssidibe/datasidibe/nouveau/concessionnaires.csv",header=True,inferSchema=True)
display(df_concessionnaires)

df_emails = spark.read.csv("/Volumes/databrickssidibe/datasidibe/nouveau/emails.csv",header=True,inferSchema=True)
display(df_emails)

df_products_data = spark.read.csv("/Volumes/databrickssidibe/datasidibe/nouveau/products_data.csv",header=True,inferSchema=True)
display(df_products_data)

df_produits = spark.read.csv("/Volumes/databrickssidibe/datasidibe/nouveau/produits.csv",header=True,inferSchema=True)
display(df_produits)

df_ventes = spark.read.csv("/Volumes/databrickssidibe/datasidibe/nouveau/ventes.csv",header=True,inferSchema=True)
display(df_ventes)

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [0]:
df_clients.write.mode("overwrite").saveAsTable("bronze.clients")
df_commerciaux.write.mode("overwrite").saveAsTable("bronze.commerciaux")
df_concessionnaires.write.mode("overwrite").saveAsTable("bronze.concessionnaires")
df_emails.write.mode("overwrite").saveAsTable("bronze.emails")
df_products_data.write.mode("overwrite").saveAsTable("bronze.products_data")
df_produits.write.mode("overwrite").saveAsTable("bronze.produits")
df_ventes.write.mode("overwrite").saveAsTable("bronze.ventes")

In [0]:
display(df_products_data)


In [0]:
from pyspark.sql.functions import lit

df_clients = df_clients.withColumn("source", lit("clients"))
df_ventes = df_ventes.withColumn("source", lit("ventes"))
df_products_data = df_products_data.withColumn("source", lit("products_data"))
df_produits = df_produits.withColumn("source", lit("produits"))
df_commerciaux = df_commerciaux.withColumn("source", lit("commerciaux"))
df_concessionnaires = df_concessionnaires.withColumn("source", lit("concessionnaires"))
df_emails = df_emails.withColumn("source", lit("emails"))

In [0]:
display(df_commerciaux)

In [0]:
from pyspark.sql.functions import when, col, trim, lower

def nettoyer(df):
    for c in df.columns:
        df = df.withColumn(
            c,
            when(trim(lower(col(c))) == "null", None)
            .when(trim(col(c)) == "", None)
            .otherwise(col(c))
        )
    return df

df_clients = nettoyer(df_clients)
df_ventes = nettoyer(df_ventes)
df_products_data = nettoyer(df_products_data)
df_produits = nettoyer(df_produits)
df_commerciaux = nettoyer(df_commerciaux)
df_concessionnaires = nettoyer(df_concessionnaires)
df_emails = nettoyer(df_emails)


In [0]:
# fusion des tables
from functools import reduce

dfs = [
    df_clients,
    df_ventes,
    df_products_data,
    df_produits,
    df_commerciaux,
    df_concessionnaires,
    df_emails
]

bronze_raw = reduce(
    lambda d1, d2: d1.unionByName(d2, allowMissingColumns=True),
    dfs
)

In [0]:
# verification des modalite de la variables source
bronze_raw.groupBy("source").count().show()

In [0]:
display(bronze_raw)



In [0]:
#placer la table bronze_raw dans bronze
bronze_raw.write.format("delta").mode("overwrite").saveAsTable(
    "databrickssidibe.bronze.bronze_raw"
)


In [0]:
schema = "bronze"

tables = spark.sql(f"SHOW TABLES IN {schema}").collect()

for t in tables:
    spark.sql(f"DROP TABLE IF EXISTS {schema}.{t.tableName}")

In [0]:
bronze_raw.columns

In [0]:

%sql
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
from pyspark.sql.functions import trim, col

In [0]:
df_silver = bronze_raw
display(df_silver)

In [0]:
for c in df_silver.columns:
    df_silver = df_silver.withColumn(
        c,
        trim(col(c))
    )

In [0]:
df_silver = df_silver.na.drop("all")

In [0]:
df_silver = df_silver.na.drop(subset=["id"])

In [0]:
df_silver.write.mode("overwrite").saveAsTable("silver.silver")

In [0]:
display(df_silver)

In [0]:
%sql
SELECT
    customer_id,
    SUM(sales_amount) AS total_ventes,
    COUNT(*) AS nb_ventes
FROM silver.silver_all
WHERE source = 'ventes'
GROUP BY customer_id;

In [0]:
df_null_cols = df.select([
    c for c in df.columns
    if df.filter(df[c].isNotNull()).count() == 0
])